In [1]:
from hana_ml import dataframe
url, port, user, pwd = "810070ba-df1a-4553-9a82-23690dc0158e.hana.demo-hc-3-haas-hc-dev.dev-aws.hanacloud.ondemand.com", \
443, "SAPUSER", "nIL0yr8S"
#url='hcp-ml-validate.hana-ml.c.ap-cn-1.cloud.sap'
#user='MLAPITESTER'
#passwd='Abcd12345'
#port=30315
cc = dataframe.ConnectionContext(url, port, user, pwd)

In [2]:
import numpy as np
import pandas as pd
np.random.seed(2025)
data = pd.concat((pd.DataFrame(dict(DATE=pd.date_range(start='1/1/2025', periods=128),
                                    ID=range(128))),
                  pd.DataFrame(np.random.normal(size=128), columns=['X'])),
                  axis=1)

In [3]:
from hana_ml.dataframe import create_dataframe_from_pandas
garch_df = create_dataframe_from_pandas(cc, data,
                                        "SIM_GARCH_DATA_TBL",
                                        force=True)

100%|██████████| 1/1 [00:00<00:00,  2.72it/s]


In [4]:
from hana_ai.tools.hana_ml_tools.garch_tools import GARCHFitPredict
garch_tool = GARCHFitPredict(cc)

In [5]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4o', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [garch_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

C:\Users\I326292\AppData\Local\Temp\ipykernel_17792\2577452375.py:5: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)


In [9]:
instruction_str = "Please predict the volatility of column X in table SIM_GARCH_DATA_TBL for the next incoming 5 days,"+\
" where key is DATE and model_type is tgarch."
agent_chain.invoke(instruction_str)

{'input': 'Please predict the volatility of column X in table SIM_GARCH_DATA_TBL for the next incoming 5 days, where key is DATE and model_type is tgarch.',
 'output': '{"garch_predict_result_table": "SIM_GARCH_DATA_TBL_PREDICT_RESULT"}'}

In [10]:
cc.table('SIM_GARCH_DATA_TBL_PREDICT_RESULT').shape

[5, 3]

In [13]:
instruction_str = "Please predict the volatility of column Y in table SIM_GARCH_DATA_TBL for the next incoming 5 days,"+\
" where key is DATE and model_type is garch."
agent_chain.invoke(instruction_str)

{'input': 'Please predict the volatility of column Y in table SIM_GARCH_DATA_TBL for the next incoming 5 days, where key is DATE and model_type is garch.',
 'output': '{"ValueError occurred": "Column(s) not in DataFrame: [\'Y\']"}'}

In [ ]:
try:
    instruction_str = "Please predict the volatility of column DATE in table SIM_GARCH_DATA_TBL for the next incoming 5 days,"+\
    " where key is ID"
    agent_chain.invoke(instruction_str)
except Exception as err:
    assert('AFL error' in str(err))

ERROR:hana_ml.algorithms.pal.tsa.garch:(423, 'AFL error: AFL DESCRIBE for nested call failed - invalid table(s) for ANY-procedure call (Input table 0: column 1 (starting with 0) has invalid SQL type.): line 9 col 1 (at pos 350)')
Traceback (most recent call last):
  File "c:\users\i326292\desktop\hanamlapi\src\hana_ml\algorithms\pal\tsa\garch.py", line 235, in fit
    self._call_pal_auto(conn,
  File "c:\users\i326292\desktop\hanamlapi\src\hana_ml\algorithms\pal\pal_base.py", line 940, in _call_pal_auto
    self.execute_statement, materialize_dict = call_pal_auto_with_hint(conn_context,
                                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\users\i326292\desktop\hanamlapi\src\hana_ml\algorithms\pal\pal_base.py", line 1448, in call_pal_auto_with_hint
    if try_exec(cur, sql, conn):
       ^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\users\i326292\desktop\hanamlapi\src\hana_ml\algorithms\pal\pal_base.py", line 1403, in try_exec
    cur.execute(sql)
hdbcli

Error: (423, 'AFL error: AFL DESCRIBE for nested call failed - invalid table(s) for ANY-procedure call (Input table 0: column 1 (starting with 0) has invalid SQL type.): line 9 col 1 (at pos 350)')

In [17]:
cc.drop_table("SIM_GARCH_DATA_TBL_PREDICT_RESULT")
cc.drop_table("SIM_GARCH_DATA_TBL")

In [18]:
cc.close()